# Language Subtitles — Generate

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Second step of Language Subtitles: redistributes each language's raw text (from
`caption-multilang-sources-gather.ipynb`) onto the **master caption**'s exact blocks and timing, using AI
(Mistral, with Groq as fallback).

**Master caption** (`config.nome_legenda_mestre`): by default, the corrected SRT from Single
Subtitle. It defines the segmentation/timing every language must follow — this is what keeps all
languages synchronized when they're burned stacked on screen later.

**Per-language source choice** (`FONTE_TEXTO_IDIOMA` below): each language can use either its
YouTube captions (`"yt"`) or its Whisper transcription (`"whisper"`) as the raw text to redistribute.
Pick whichever you already corrected / trust more for that language.

The AI is asked for exactly N parts (N = number of master blocks). If it returns a different
count, this notebook **always saves anyway** — it adjusts automatically (merges extra parts, or
splits long ones) rather than silently discarding the result. Lines that couldn't be
auto-adjusted are marked `[revisar manualmente]` so you can spot them at a glance.

**Manual correction workflow** (same pattern as before): download the final SRTs, correct
locally, upload back to Drive — `caption-multilang-burn.ipynb` always reads the current file.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

!apt-get -qq -y install ffmpeg > /dev/null 2>&1
print('✅ ffmpeg')

!pip install -q openai
print('✅ openai (client library — used for both Mistral and Groq APIs)')

from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=False)
except Exception:
    pass

# ── Clear stale local data files from any previous run in this session ─────
# Modules above are always freshly copied (rmtree + copytree), but DATA files
# (.srt, .wav, .ass) downloaded/generated by earlier cells in this same
# session could still be sitting in /content — if you re-run after
# correcting something on Drive, you want THIS run to fetch everything
# fresh, not silently reuse an old local copy. This removes any leftover
# language-subtitle data files before starting.
padroes_para_limpar = ["*.srt", "*.wav", "*.ass"]
removidos = 0
for padrao in padroes_para_limpar:
    for arquivo in Path('/content').glob(padrao):
        arquivo.unlink()
        removidos += 1
print(f"✅ {removidos} stale local file(s) cleared — this run will fetch everything fresh from Drive")

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-24s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell — same NOME_ORACAO as earlier steps     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY ────────────────────────────────────────────────────────
NOME_ORACAO = "40_Matt_02"

# ── 2. MASTER LANGUAGE (must match caption-single-generate.ipynb / video-base-*.ipynb) ──
# The narration's own language — defines which "_whisper_{lang}.srt" is the
# default master. Required here even though NOME_LEGENDA_MESTRE below is set
# explicitly, for consistency with the rest of the config.
IDIOMA_MESTRE = "en"

# ── 3. TARGET LANGUAGES (must match caption-multilang-sources-gather.ipynb) ─────────
IDIOMAS_ALVO = ["pt", "es", "fr", "ko"]

# ── 4. PER-LANGUAGE RAW TEXT SOURCE ─────────────────────────────────────────
# "yt" = legenda do YouTube · "whisper" = transcrição do áudio dublado.
#
# O padrão é "yt", e a razão é medida. No 40_Matt_02, contando os nomes
# próprios que TÊM que aparecer no capítulo:
#
#            YouTube   Whisper
#     pt      19/19     14/19
#     es      19/19     14/19
#     fr      19/19     13/19
#     ko      18/18     10/18
#
# O Whisper das faixas dubladas escreveu "Eodes" por Herodes, "Jérim Hala" por
# Jérémie, 몰약(mirra) virou 모략(estratagema), e o português começou com uma
# frase que não existe no capítulo.
#
# O comentário anterior aqui recomendava "whisper" com o argumento de que a
# legenda deve casar com o que se ouve. Não se aplica: a queima usa só o vídeo
# base e as legendas -- o áudio dublado NUNCA é tocado. O espectador ouve a
# narração em inglês, e para ela o mestre já é o whisper_en corrigido à mão.
# Fidelidade à dublagem não compra nada aqui, e custa os nomes próprios.
#
# Só troque para "whisper" se você tiver corrigido o whisper_{lang}.srt à mão.
FONTE_TEXTO_IDIOMA = {
    "pt": "yt",
    "es": "yt",
    "fr": "yt",
    "ko": "yt",
}

# ── 5. MASTER CAPTION (manual) ───────────────────────────────────────────────
# The corrected whisper_en.srt — defines segmentation/timing for every language.
# Filled in manually (not left blank) so it's explicit and doesn't silently
# depend on IDIOMA_MESTRE's default elsewhere.
NOME_LEGENDA_MESTRE = "40_Matt_02_whisper_en.srt"

# ── 6. DRIVE ROOT FOLDER ────────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project

# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION")
print("=" * 60)
print(f"   Video:            {NOME_ORACAO}")
print(f"   Master language:  {IDIOMA_MESTRE}")
print(f"   Target langs:     {IDIOMAS_ALVO}")
print(f"   Sources:          {FONTE_TEXTO_IDIOMA}")
print(f"   Master caption:   {NOME_LEGENDA_MESTRE or '(default — Single Subtitle SRT)'}")
print(f"   Drive root:       {PASTA_DRIVE_RAIZ}")
print("=" * 60)
print("✅ Configuration ready — proceed to Initialization")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ║  Needs MISTRAL_KEY and/or GROQ_KEY set in Colab Secrets (🔑 icon ║
# ║  on the left sidebar) — at least one is required.                ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path
from google.colab import userdata

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from language_captions_pipeline import LanguageCaptionsPipeline
from groq_client import GroqClient

config = PipelineConfig(
    NOME_ORACAO          = NOME_ORACAO,
    PASTA_DRIVE_RAIZ      = PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE         = IDIOMA_MESTRE,
    FONTE_TEXTO_IDIOMA    = FONTE_TEXTO_IDIOMA,
    NOME_LEGENDA_MESTRE   = NOME_LEGENDA_MESTRE,
)

try:
    mistral_key = userdata.get('MISTRAL_KEY')
except Exception:
    mistral_key = ''
try:
    groq_key = userdata.get('GROQ_KEY')
except Exception:
    groq_key = ''

groq_client = GroqClient.get(mistral_key=mistral_key or '', groq_key=groq_key or '', nome_oracao=config.NOME_ORACAO)
pipeline = LanguageCaptionsPipeline(config, groq_client=groq_client)

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:            {config.NOME_ORACAO}")
print(f"   Folder:           {config.pasta_oracao}")
print(f"   Master caption:   {config.nome_legenda_mestre}")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🤖 REDISTRIBUTE — AI splits each language onto the master's blocks ║
# ║  Always saves a result per successful language — auto-adjusts if ║
# ║  the AI returns a different count than expected (never silently  ║
# ║  discards). Takes a few seconds per language (rate-limit delay). ║
# ╚══════════════════════════════════════════════════════════════════╝

resultado = pipeline.redistribuir_idiomas(IDIOMAS_ALVO)
print(f"\n✅ {len(resultado)}/{len(IDIOMAS_ALVO)} languages generated: {list(resultado.keys())}")

faltando = [l for l in IDIOMAS_ALVO if l not in resultado]
if faltando:
    print(f"⚠️  Skipped (see warnings above): {faltando}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW — final per-language subtitles                       ║
# ║  Lines marked ⚠️ [revisar manualmente] need your attention.       ║
# ╚══════════════════════════════════════════════════════════════════╝

for lang, legendas in resultado.items():
    print(f"\n--- {lang.upper()} ({len(legendas)} blocks) ---")
    for leg in legendas:
        marca = " ⚠️" if "[revisar manualmente]" in leg.texto else ""
        print(f"  [{leg.inicio_str} → {leg.fim_str}]  {leg.texto}{marca}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — for manual correction                             ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files

if not resultado:
    print("Nothing to download — no languages were generated.")
else:
    for lang in resultado:
        caminho = Path(config.nome_srt(lang))
        print(f"📥 {caminho.name}")
        files.download(str(caminho))

    print()
    print("After correcting locally, upload the files back to Drive at:")
    print(f"   {config.pasta_oracao}")
    print("(overwrite the same filenames)")
